> **Use note:** This is an earlier optional notebook. It combines topics that now belong to several course weeks. It does not replace the current weekly main and support files or define current report requirements. All data and results in this notebook are artificial practice evidence.
>
> **使用說明：**這是較早版本的選用 notebook，合併了目前分屬多個週次的內容。它不取代各週現行的 main 與 support 檔案，也不規定目前的報告要求。檔案中的資料與結果都是練習用人工證據。


# 01 TypeB 上機教材：從 AI 投資故事到可檢驗交易問題

**Big HOT：什麼時候一個 AI 產生的投資看法，才算是可以被資料檢驗的交易問題？**

這份 notebook 不是先教模型，而是先訓練學生把「我覺得 AI 可以預測漲跌」改寫成可檢驗、可回測、可被反駁的策略研究問題。

3 小時課堂只跑「核心 HOT 主線」。Optional HOT 與 Appendix 留給快班、課後或你挑選替換。

**本堂 learning evidence**

- 一句判斷：AI 預測為什麼還不是策略
- 資料可得性與時間戳分類表
- 策略假說與反證條件
- 小組策略問題卡


## 課前閱讀與 GitHub 回應

課前請先閱讀 [Week 2 main](../../02/week2_main.ipynb)。

課前 HOT：

```text
AI says: "This ETF will rise tomorrow."
Would you trade? Why or why not?
```

GitHub 回應格式：

```text
My initial answer:
I would / would not trade because ______.

Missing conditions:
1.
2.
3.

One failure condition:
This strategy fails if ______.
```

課中使用方式：開場先比較課前答案，再用本 notebook 的 timing audit、benchmark 與 strategy problem card 修正初始判斷。


## 3 小時 HOT storyline

| 時間 | Cluster | 核心 HOT | 可見產出 |
|---:|---|---|---|
| 0:00-0:25 | C1. Prediction is not strategy | AI 說明天會漲 2%，你敢下單嗎？缺什麼？ | 判斷句 |
| 0:25-0:55 | C1. AI roles | 哪些金融任務適合 AI 支援，哪些不該交給 AI 決策？ | 任務分類表 |
| 0:55-1:30 | C2. Data and target | 同一個想法，換成日、週、月資料，研究問題會怎樣改變？ | target 選擇 |
| 1:30-2:05 | C2. Information timing | 哪些欄位在交易當下真的已知？ | leakage 風險表 |
| 2:05-2:35 | C3. Evidence and benchmark | 什麼證據會讓你放棄這個策略？ | 反證條件 |
| 2:35-3:00 | C4. Strategy problem card | 把故事改成一張可回測的策略問題卡 | 小組卡片 |

**Optional HOT**：交易成本、benchmark、AI prompt critique、同儕質詢。  
**Appendix HOT**：financial ML failure、event bars、triple-barrier、meta-labeling。


## Learning Loop Map（新版 Type B 操作版）

| Loop | Mini-input | BIT / HOT | Visible output | Delayed feedback focus |
|---|---|---|---|---|
| 1 | AI prediction vs strategy | Predict + justify | 判斷句與 missing conditions | prediction 和 trading rule 的差異 |
| 2 | AI roles in finance | Classify | AI task classification table | decision 不應完全交給 AI |
| 3 | Data frequency and target | Compare + transform | data frequency / target 表 | frequency 如何改變 turnover 與 target |
| 4 | Information timing | Classify + debug | timing audit table | 下單前已知 vs future information |
| 5 | Benchmark and falsification | Compare / falsify | benchmark 解讀與 failure condition | 沒有 benchmark 不能說策略有效 |
| 6 | Strategy problem design | Synthesize | strategy problem card | 可檢驗、可回測、可反駁 |


## TypeB 課堂語言

學生可用句型：

```text
I would / would not trade because ______.
The missing condition is ______.
This is not testable yet because ______.
The data is known before trade / after trade / unclear because ______.
The strategy fails if ______.
```

教師延後回饋句：

```text
What evidence would change your mind?
Which part is a prediction, and which part is a trading decision?
What exactly must be measured before you can test that?
```


## 學生回應方式（課中使用）

本堂課的回應不求長，但每次都要有 claim + reason + missing evidence。

| 場景 | 回應格式 |
|---|---|
| Big HOT | `I would / would not trade because ______.` |
| Timing audit | `This data is known before trade / future / unclear because ______.` |
| Benchmark | `The benchmark matters because ______.` |
| Strategy card | `Our strategy is testable because ______.` |

小組分享規則：

```text
Choose one spokesperson.
Show one table or one sentence.
Explain one reason, not only the answer.
```


In [ ]:
# 課堂穩定性設定：預設使用合成 OHLCV 資料。
# 若教室網路穩定且已安裝 yfinance，可以把 USE_ONLINE_DATA 改成 True。
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True

SYMBOL = "0050.TW"
USE_ONLINE_DATA = False


def make_synthetic_ohlcv(n=760, seed=42):
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range("2021-01-01", periods=n)
    t = np.arange(n)
    regime = np.select(
        [t < n * 0.33, t < n * 0.66],
        [0.00045, -0.00015],
        default=0.00025,
    )
    shocks = rng.normal(0, 0.011, n)
    shocks[0] = rng.normal(0, 0.011)
    ret = regime + shocks + 0.08 * np.r_[0, shocks[:-1]]
    close = 100 * np.exp(np.cumsum(ret))
    open_ = close * (1 + rng.normal(0, 0.003, n))
    high = np.maximum(open_, close) * (1 + rng.uniform(0.001, 0.012, n))
    low = np.minimum(open_, close) * (1 - rng.uniform(0.001, 0.012, n))
    volume = rng.lognormal(mean=15.2, sigma=0.25, size=n) * (1 + 8 * np.abs(ret))
    df = pd.DataFrame(
        {
            "Open": open_,
            "High": high,
            "Low": low,
            "Close": close,
            "Adj Close": close,
            "Volume": volume.astype(int),
        },
        index=dates,
    )
    df.index.name = "Date"
    return df


def load_market_data(symbol=SYMBOL, start="2020-01-01", use_online=USE_ONLINE_DATA):
    if use_online:
        try:
            import yfinance as yf
            df = yf.download(symbol, start=start, auto_adjust=False, progress=False)
            if isinstance(df.columns, pd.MultiIndex):
                df.columns = df.columns.get_level_values(0)
            if not df.empty and {"Open", "High", "Low", "Close", "Volume"}.issubset(df.columns):
                print(f"Loaded online data: {symbol}, rows={len(df)}")
                return df.dropna()
        except Exception as exc:
            print("Online download failed; falling back to synthetic data.")
            print(type(exc).__name__, exc)
    print("Using synthetic OHLCV data. Toggle USE_ONLINE_DATA=True for real market data.")
    return make_synthetic_ohlcv()


def add_features(raw):
    df = raw.copy()
    df["ret_1d"] = df["Close"].pct_change()
    df["ret_fwd_1d"] = df["Close"].shift(-1) / df["Close"] - 1
    df["ma_5"] = df["Close"].rolling(5).mean()
    df["ma_20"] = df["Close"].rolling(20).mean()
    df["ma_gap"] = df["ma_5"] / df["ma_20"] - 1
    df["mom_5"] = df["Close"] / df["Close"].shift(5) - 1
    df["mom_20"] = df["Close"] / df["Close"].shift(20) - 1
    df["vol_20"] = df["ret_1d"].rolling(20).std() * np.sqrt(252)
    df["range_pct"] = (df["High"] - df["Low"]) / df["Close"]
    volume_mean = df["Volume"].rolling(20).mean()
    volume_std = df["Volume"].rolling(20).std()
    df["volume_z"] = (df["Volume"] - volume_mean) / volume_std
    df = df.dropna()
    df["target_up"] = (df["ret_fwd_1d"] > 0).astype(int)
    return df


def max_drawdown(ret):
    wealth = (1 + ret.fillna(0)).cumprod()
    dd = wealth / wealth.cummax() - 1
    return dd.min()


def sharpe(ret, periods=252):
    vol = ret.std()
    if vol == 0 or np.isnan(vol):
        return np.nan
    return np.sqrt(periods) * ret.mean() / vol


def perf_table(returns_dict):
    rows = []
    for name, ret in returns_dict.items():
        ret = pd.Series(ret).dropna()
        rows.append(
            {
                "strategy": name,
                "ann_return": (1 + ret).prod() ** (252 / len(ret)) - 1 if len(ret) else np.nan,
                "ann_vol": ret.std() * np.sqrt(252),
                "sharpe": sharpe(ret),
                "max_drawdown": max_drawdown(ret),
                "win_rate": (ret > 0).mean(),
            }
        )
    return pd.DataFrame(rows).set_index("strategy").round(4)


raw = load_market_data()
df = add_features(raw)
df.tail()


## C1 HOT 1：AI 說明天會漲 2%，這是不是策略？

### Type B 操作標籤

| 項目 | 設計 |
|---|---|
| BIT type | Predict + justify |
| Think | 60 秒：先選可交易 / 不可交易 / 資訊不足，寫一個理由。 |
| Pair/Group | 兩人比較缺少哪些交易條件。 |
| Visible output | 一句判斷 + 三個 missing conditions。 |
| Delayed feedback | 先請兩組比較答案，再追問：What else must be defined before trading? |

先不要寫程式。請學生先投票：**可交易 / 不可交易 / 資訊不足**。

Think：你會問 AI 哪三個追問？  
Pair：把追問分類成資料、交易規則、風險、成本。  
Share：每組只講一個「如果沒有它就不能交易」的條件。


In [ ]:
hot1_cases = pd.DataFrame(
    [
        ["AI says price will rise tomorrow", "prediction", "No position size, no exit, no cost, no benchmark"],
        ["Buy when probability_up > 0.55 and exit next day", "rule", "Need evidence that 0.55 works out-of-sample"],
        ["Hold 0050.TW forever", "benchmark", "Not AI, but useful comparison"],
        ["Ask GenAI for five stock ideas", "idea support", "Needs data, timestamp, validation, risk control"],
    ],
    columns=["statement", "best_class", "missing_for_trading"],
)
hot1_cases


## C1 HOT 2：哪些任務適合 AI，哪些必須由人決策？

### Type B 操作標籤

| 項目 | 設計 |
|---|---|
| BIT type | Classify |
| Think | 60 秒：個人先把任務分成 support / prediction / decision / risk。 |
| Pair/Group | 小組完成分類表，標出最危險的誤分。 |
| Visible output | AI task classification table。 |
| Delayed feedback | 請不同組比較 decision 與 risk review 的界線。 |

HOT 類型：**Classify + Justify**  
任務：把下列工作放到四欄：`support`、`prediction`、`decision`、`risk review`。

TypeB 重點不是找標準答案，而是要求學生說明「錯分的風險」。


In [ ]:
ai_task_cards = pd.DataFrame(
    {
        "task": [
            "summarize market news",
            "predict next-day return direction",
            "choose how much capital to allocate",
            "detect unusually high turnover",
            "write a trading hypothesis",
            "approve live deployment",
            "explain model feature importance",
            "generate alternative explanations for a signal",
        ],
        "student_class": "",
        "risk_if_wrong": "",
    }
)
ai_task_cards


## C2 HOT 3：同一個想法，資料頻率改變後，問題還一樣嗎？

### Type B 操作標籤

| 項目 | 設計 |
|---|---|
| BIT type | Compare + transform |
| Think | 60 秒：先預測日、週、月資料會改變什麼。 |
| Pair/Group | 小組選一種頻率，寫 target 與 holding period。 |
| Visible output | data frequency -> target -> risk 表。 |
| Delayed feedback | 追問：How does turnover change when frequency changes? |

Big HOT 往下推：如果我們要檢驗「近期上漲可能延續」，用日資料、週資料、月資料會得到同一種策略嗎？

HOT 類型：**Compare + Transform**  
可見產出：每組選一個時間尺度，寫出 target、holding period、可能成本。


In [ ]:
idea_versions = pd.DataFrame(
    [
        ["daily", "predict next-day direction", "1 trading day", "higher turnover, cost matters"],
        ["weekly", "predict next-week return", "5 trading days", "fewer trades, slower feedback"],
        ["monthly", "predict next-month relative performance", "20 trading days", "closer to asset allocation"],
    ],
    columns=["data_frequency", "possible_target", "holding_period", "main_risk"],
)
idea_versions


In [ ]:
ax = raw["Close"].plot(title="Price path: visual inspection before modeling")
ax.set_ylabel("Close")
plt.show()

ax = df["ret_1d"].plot(title="Daily returns: strategy evidence lives in returns, not only price levels")
ax.set_ylabel("daily return")
plt.show()


## C2 HOT 4：哪些資料在下單當下真的已知？

### Type B 操作標籤

| 項目 | 設計 |
|---|---|
| BIT type | Classify + debug |
| Think | 60 秒：個人標記 known / future / unclear。 |
| Pair/Group | 小組找出最可能造成 leakage 的欄位。 |
| Visible output | timing audit table。 |
| Delayed feedback | 先讓兩組比較 unclear 欄位，再補 timestamp 概念。 |

HOT 類型：**Classify + Debug**  
請學生把欄位分成：`known_before_trade`、`known_after_trade`、`unclear_need_rule`。

這題銜接下一堂課的 leakage：金融 ML 最容易犯的錯，不是模型太弱，而是資料時間線不乾淨。


In [ ]:
timing_cards = pd.DataFrame(
    {
        "field_or_feature": [
            "today open",
            "today close",
            "yesterday close",
            "20-day moving average using yesterday close",
            "tomorrow return",
            "full-sample mean return",
            "news published before market open",
            "revised financial statement data",
        ],
        "student_label": "",
        "why": "",
    }
)
timing_cards


## C3 HOT 5：什麼證據會讓你放棄這個策略？

### Type B 操作標籤

| 項目 | 設計 |
|---|---|
| BIT type | Falsify + rank |
| Think | 60 秒：先寫一個會讓你放棄策略的結果。 |
| Pair/Group | 小組把 failure conditions 排序。 |
| Visible output | 可觀察的 failure condition。 |
| Delayed feedback | 追問：Is this condition measurable before deployment? |

HOT 類型：**Falsify + Rank**  
請學生不要只寫「如果賠錢」。要寫能在研究階段檢查的失敗條件。

範例：若扣除 0.15% 單邊成本後 Sharpe 接近 0，策略失敗。  
範例：若優勢只出現在訓練期、測試期消失，策略失敗。


In [ ]:
falsification_template = pd.DataFrame(
    {
        "claim": [
            "Recent winners keep winning",
            "AI probability improves timing",
            "Volume confirms trend",
            "The strategy is robust",
        ],
        "evidence_that_supports": "",
        "evidence_that_refutes": "",
        "minimum_benchmark": ["buy-and-hold", "naive momentum", "cash", "same strategy with higher cost"],
    }
)
falsification_template


## C3 HOT 6：沒有 benchmark 的高報酬，能不能說策略有效？

### Type B 操作標籤

| 項目 | 設計 |
|---|---|
| BIT type | Compare |
| Think | 60 秒：先判斷策略是否真的優於 benchmark。 |
| Pair/Group | 小組引用圖或表支持判斷。 |
| Visible output | benchmark interpretation sentence。 |
| Delayed feedback | 請另一組挑戰：What does benchmark reveal that raw return hides? |

HOT 類型：**Compare + Justify**  
學生先看兩條 equity curve，再回答：哪一條比較值得研究？為什麼？


In [ ]:
benchmark = df["ret_1d"]
simple_mom = np.where(df["mom_20"] > 0, 1, 0) * df["ret_fwd_1d"]
curves = pd.DataFrame(
    {
        "buy_and_hold": (1 + benchmark).cumprod(),
        "simple_momentum_rule": (1 + pd.Series(simple_mom, index=df.index)).cumprod(),
    }
)
curves.plot(title="First benchmark check: a strategy must beat something meaningful")
plt.show()

perf_table({"buy_and_hold": benchmark, "simple_momentum_rule": pd.Series(simple_mom, index=df.index)})


## C4 HOT 7：AI 產生的策略構想，哪些句子要保留、刪除、修改？

### Type B 操作標籤

| 項目 | 設計 |
|---|---|
| BIT type | Critique + transform |
| Think | 60 秒：個人標出 keep / delete / revise。 |
| Pair/Group | 小組重寫一個過度模糊或過度自信句子。 |
| Visible output | AI critique table。 |
| Delayed feedback | 延後回饋時先比較兩種改寫，最後才收斂語氣。 |

HOT 類型：**Critique + Transform**  
教師可以把下列表格當成 GenAI 輸出的替身，要求學生把模糊語句改成可驗證語句。


In [ ]:
ai_strategy_text = pd.DataFrame(
    [
        ["Use AI to find good buying points", "revise", "Define signal, target, horizon, benchmark"],
        ["Buy when recent momentum is strong", "revise", "Define lookback window and threshold"],
        ["This should make stable profit", "delete", "Overclaim; replace with testable performance criteria"],
        ["Compare against buy-and-hold after transaction costs", "keep", "This is testable"],
        ["Avoid overfitting by checking a later test period", "keep", "This is validation-aware"],
    ],
    columns=["AI_sentence", "student_action", "reason"],
)
ai_strategy_text


## C4 HOT 8：完成策略問題卡

### Type B 操作標籤

| 項目 | 設計 |
|---|---|
| BIT type | Synthesize |
| Think | 90 秒：個人先填三欄：target、benchmark、failure condition。 |
| Pair/Group | 小組完成策略問題卡。 |
| Visible output | strategy problem card。 |
| Delayed feedback | 請同儕只問 evidence-based questions，再給教師總結。 |

這是本堂最重要的可見產出。  
要求：不是完整策略，而是一個可以帶到 Week 3 做資料與特徵工程的研究問題。


In [ ]:
strategy_problem_card = pd.DataFrame(
    {
        "field": [
            "Trading idea",
            "Asset / universe",
            "Prediction target",
            "Holding period",
            "Features needed",
            "Benchmark",
            "Transaction cost assumption",
            "Failure condition",
            "Human decision still required",
        ],
        "group_answer": [
            "Momentum may predict short-term continuation",
            "0050.TW or selected ETF universe",
            "next-day or next-week direction/return",
            "1-5 trading days",
            "past returns, moving average gap, volatility, volume",
            "buy-and-hold and naive momentum",
            "0.05%-0.15% per trade",
            "no improvement after cost in test period",
            "capital allocation and deployment approval",
        ],
    }
)
strategy_problem_card


## Optional HOT：3 小時內時間不夠時可跳過

1. **Cost predict**：哪一種策略最怕交易成本，為什麼？
2. **Prompt compare**：模糊 prompt 與可驗證 prompt 產出的策略差在哪？
3. **Peer challenge**：別組只能問 evidence-based questions，不准直接說好或不好。
4. **Rank**：把資料問題、模型問題、成本問題、風險問題依「最可能害策略失敗」排序。


In [ ]:
optional_cost_cards = pd.DataFrame(
    [
        ["trade every day", 0.95, "very high"],
        ["trade only when probability > 0.60", 0.35, "medium"],
        ["monthly rebalance", 0.06, "low"],
        ["buy and hold", 0.01, "very low"],
    ],
    columns=["rule", "approx_turnover", "cost_sensitivity"],
)
optional_cost_cards


## Appendix HOT A1：金融 ML 為什麼常失敗？

HOT 類型：**Rank + Justify**  
把失敗原因依「初學者最容易犯」排序，並寫出防範方法。


In [ ]:
failure_modes = pd.DataFrame(
    [
        ["data leakage", "using information unavailable at trade time", "timestamp audit"],
        ["overfitting", "choosing rules after seeing test results", "holdout and walk-forward validation"],
        ["ignored costs", "profitable before cost only", "cost sensitivity table"],
        ["weak benchmark", "beating cash but not buy-and-hold", "define benchmark before testing"],
        ["non-stationarity", "pattern disappears in new regime", "subperiod and regime tests"],
    ],
    columns=["failure_mode", "meaning", "prevention"],
)
failure_modes


## Appendix HOT A2：Calendar bars 一定是最好的資料切法嗎？

HOT 類型：**Compare**  
如果市場有些日子成交很熱、有些日子很冷，固定一天一筆資料是否公平？


In [ ]:
event_df = df.copy()
event_df["dollar_volume"] = event_df["Close"] * event_df["Volume"]
threshold = event_df["dollar_volume"].quantile(0.80)
event_bars = event_df[event_df["dollar_volume"] >= threshold]
pd.DataFrame(
    {
        "calendar_rows": [len(event_df)],
        "event_rows_top_20pct_dollar_volume": [len(event_bars)],
        "avg_calendar_abs_return": [event_df["ret_1d"].abs().mean()],
        "avg_event_abs_return": [event_bars["ret_1d"].abs().mean()],
    }
).round(4)


## Appendix HOT A3：固定 horizon 標籤一定合理嗎？

HOT 類型：**Transform**  
把「明天漲跌」改成：先碰到上界、下界，或時間到期。


In [ ]:
def simple_triple_barrier(close, horizon=10, up=0.03, down=-0.03):
    labels = []
    for i in range(len(close) - horizon):
        path = close.iloc[i + 1 : i + horizon + 1] / close.iloc[i] - 1
        hit_up = path[path >= up]
        hit_down = path[path <= down]
        if len(hit_up) == 0 and len(hit_down) == 0:
            labels.append(0)
        elif len(hit_down) == 0 or (len(hit_up) and hit_up.index[0] < hit_down.index[0]):
            labels.append(1)
        else:
            labels.append(-1)
    return pd.Series(labels, index=close.index[: len(labels)])

tb_label = simple_triple_barrier(df["Close"])
tb_label.value_counts().rename("count")


## Appendix HOT A4：Meta-labeling 為什麼把 side 和 size 分開？

HOT 類型：**Explain + Design**  
第一個模型決定方向，第二個模型決定這次要不要真的交易。這和單一模型有什麼差異？


In [ ]:
meta_label_demo = pd.DataFrame(
    {
        "side_model_signal": ["long", "long", "long", "flat"],
        "confidence_or_filter": [0.62, 0.51, 0.74, 0.40],
        "meta_decision": ["trade", "skip", "trade", "skip"],
        "reason": [
            "signal passes threshold",
            "edge too weak after cost",
            "strong enough but still needs risk limit",
            "no directional view",
        ],
    }
)
meta_label_demo


## Learning Evidence Checklist

本堂課結束前，至少留下這些 evidence：

- [ ] 一句判斷：AI prediction 為什麼還不是完整 strategy。
- [ ] AI task classification table。
- [ ] timing audit table。
- [ ] benchmark interpretation sentence。
- [ ] 可觀察的 failure condition。
- [ ] strategy problem card。

Exit ticket：

```text
My strategy hypothesis is ______.
This strategy fails if ______.
The evidence I still need is ______.
```


## 參考資料

本課程設計參考下列概念來源，重點不是要求學生讀完整篇，而是把研究中的核心判斷轉成上機問題。

- López de Prado, M. (2018). *Advances in Financial Machine Learning*. 用於 financial ML failure、labeling、triple-barrier、meta-labeling、finance cross-validation、backtest overfitting。
- Gu, S., Kelly, B., & Xiu, D. (2020). Empirical Asset Pricing via Machine Learning. *Review of Financial Studies*. 用於 momentum、liquidity、volatility、非線性模型與資產報酬預測。
- Machine Learning and Portfolio Optimization 相關文獻。用於 regularization、cross-validation、estimation error 與 portfolio construction。
- Robust perspective on transaction costs in portfolio optimization 相關技術筆記。用於 transaction cost、turnover、robustness。
- XAI in finance 綜述文獻。用於 feature importance、SHAP、trust、risk assessment、governance。
- Lee, W.-Y. momentum-based sentiment trading strategy 相關研究。用於 Appendix 中 momentum + sentiment、benchmark、transaction cost、long-horizon evaluation。
- Géron, A. *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow*. 用於 practical ML workflow、train/test、validation、classification metrics、error analysis。
